# 03 - Baseline Experiments (GMM & SVM)

Train and evaluate classical ML baselines for speaker identification.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
from collections import defaultdict
from tqdm import tqdm

from src.config import load_config
from src.data.download import prepare_dataset
from src.data.splits import create_splits, get_split_metadata
from src.data.dataset import SpeakerDatasetForBaseline
from src.models.baseline_gmm import GMMBaseline
from src.models.baseline_svm import SVMBaseline
from src.training.metrics import compute_accuracy, compute_confusion_matrix
from src.evaluation.visualization import plot_confusion_matrix
from src.utils import set_seed

set_seed(42)

In [ ]:
# Load config and data
config = load_config('../configs/baseline_gmm.yaml')
metadata = prepare_dataset(config.data.data_dir, config.data.num_speakers, config.data.min_utterances_per_speaker)
splits = create_splits(metadata, config.data)

train_meta = get_split_metadata(metadata, splits, 'train')
test_meta = get_split_metadata(metadata, splits, 'test')

train_ds = SpeakerDatasetForBaseline(train_meta, config.features, config.audio)
test_ds = SpeakerDatasetForBaseline(test_meta, config.features, config.audio)

In [ ]:
# Collect features
def collect_features(dataset):
    features = defaultdict(list)
    for i in tqdm(range(len(dataset))):
        mfcc, spk_id = dataset[i]
        features[spk_id].append(mfcc)
    return dict(features)

train_features = collect_features(train_ds)
test_features = collect_features(test_ds)
print(f"Train speakers: {len(train_features)}, Test speakers: {len(test_features)}")

In [ ]:
# Train and evaluate GMM
gmm = GMMBaseline(n_components=64, use_ubm=True, ubm_components=128)
gmm_features = {spk: np.concatenate([m.T for m in mfccs], axis=0) for spk, mfccs in train_features.items()}
gmm.fit(gmm_features)

gmm_preds, gmm_labels = [], []
for spk, mfccs in tqdm(test_features.items()):
    for mfcc in mfccs:
        gmm_preds.append(gmm.predict(mfcc.T))
        gmm_labels.append(spk)

gmm_acc = compute_accuracy(np.array(gmm_preds), np.array(gmm_labels))
print(f"GMM-UBM Accuracy: {gmm_acc:.4f} ({gmm_acc*100:.1f}%)")

In [ ]:
# Train and evaluate SVM
svm = SVMBaseline(kernel='rbf', C=10.0)
svm.fit(train_features)

svm_preds, svm_labels = [], []
for spk, mfccs in tqdm(test_features.items()):
    for mfcc in mfccs:
        svm_preds.append(svm.predict(mfcc))
        svm_labels.append(spk)

svm_acc = compute_accuracy(np.array(svm_preds), np.array(svm_labels))
print(f"SVM Accuracy: {svm_acc:.4f} ({svm_acc*100:.1f}%)")

In [ ]:
# Summary
print(f"\n{'='*40}")
print(f"GMM-UBM Accuracy: {gmm_acc*100:.1f}%")
print(f"SVM Accuracy:     {svm_acc*100:.1f}%")
print(f"{'='*40}")